#Imporing Packages

In [ ]:
import torch
import torchvision
import pandas as pd
import torchvision.transforms as transforms
from torch.utils.data import DataLoader,Dataset,random_split
import torch.optim as optim
import torch.nn as nn
import os
import copy
from PIL import Image
from torchvision import models as tv_models


#Collecting Raw data

In [ ]:
import os
os.environ['KAGGLE_API_TOKEN'] = 'KGAT_0f4f9487fd8f8b05738a0af71fbb9009'

In [ ]:
!kaggle competitions download -c dog-breed-identification

100% 691M/691M [00:08<00:00, 88.1MB/s]



In [ ]:
!unzip -q dog-breed-identification.zip -d data

In [ ]:
import os

print(os.listdir("data"))

['labels.csv', 'test', 'train', 'sample_submission.csv']


In [ ]:

base_path = "/content/data"

df = pd.read_csv(os.path.join(base_path, "labels.csv"))

id_to_idx = dict(zip(df['id'], df['breed']))

image_dir = os.path.join(base_path, "train")

classes = sorted(df['breed'].unique())
class_to_idx = {cls: i for i, cls in enumerate(classes)}

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


#Data Preprocessing

In [ ]:
bad_images = []
for img_name in os.listdir(image_dir):
    path = os.path.join(image_dir,img_name)
    try:
        img = Image.open(path)
        if img.size[0] < 32 or img.size[1] < 32:
            bad_images.append(path)
            continue
        if img.mode != "RGB":
            img = img.convert("RGB")
    except Exception as e:
        bad_images.append(path)
print(f"Found {len(bad_images)} bad Images ")
for path in bad_images:
    os.remove(path)

Found 0 bad Images 


In [ ]:

class TrainDogData(Dataset):
    def __init__(self,id_to_idx,cls_to_idx,image_dir,transform=None):
        self.transform = transform
        self.id_to_idx = id_to_idx
        image_dir = image_dir
        self.paths = []
        self.labels = []
        self.class_to_idx = cls_to_idx

        for img_name in sorted(os.listdir(image_dir)):
            img_id = img_name.split(".")[0]
            label_name = id_to_idx[img_id]
            label = class_to_idx[label_name]

            self.paths.append(os.path.join(image_dir,img_name))
            self.labels.append(label)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self,idx):

        img_path = self.paths[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        label = self.labels[idx]
        return image,label

In [ ]:
val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406],std=[0.229,0.224,0.225])
])
train_transform = transforms.Compose([

    transforms.RandomResizedCrop(224,scale=(0.7,1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2,contrast=0.2,saturation=0.2,hue=0.1),
    transforms.RandomRotation(25),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406],
                         std=[0.229,0.224,0.225])
])

In [ ]:
dataset = TrainDogData(id_to_idx,class_to_idx,image_dir,train_transform)
train_size = int(0.8*len(dataset))
val_size = len(dataset) - train_size
train_dataset,val_dataset = random_split(dataset,[train_size,val_size])
train_loader = DataLoader(train_dataset,batch_size=32,shuffle=True)
val_loader = DataLoader(val_dataset,batch_size=32,shuffle=False)

#Modeling

In [ ]:
model = tv_models.resnet50(pretrained=True)

for param in model.parameters():
    param.requires_grad = False
for param in model.layer4.parameters():
    param.requires_grad = True
original_fc_layer = model.fc
num_features = original_fc_layer.in_features
num_classes = 120
new_fc_layer = nn.Linear(num_features,num_classes)
model.fc = new_fc_layer
model = model.to(device)

criterion = nn.CrossEntropyLoss()


#Train and Eval

In [ ]:

optimizer = optim.Adam([{
    'params':model.fc.parameters(),
    'lr':1e-3,
    'weight_decay': 1e-4
},                        {
                            'params':model.layer4.parameters(),
                            'lr':1e-4,
                            'weight_decay':1e-4
                        }
])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',        # because you're monitoring val_loss
    factor=0.1,        # reduce LR by 10x
    patience=2,        # wait 2 epochs before reducing

)

In [ ]:
def train_eval_model(model, train_loader, val_loader, criterion, optimizer,scheduler):

    # TRAIN
    model.train()
    train_correct = 0
    train_total = 0
    train_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        preds = outputs.argmax(dim=1)
        train_total += labels.size(0)
        train_correct += preds.eq(labels).sum().item()

    train_acc = 100. * train_correct / train_total

    # VALIDATION
    model.eval()
    val_correct = 0
    val_total = 0
    val_loss = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            preds = outputs.argmax(dim=1)
            loss = criterion(outputs,labels)
            val_loss += loss.item()
            val_total += labels.size(0)
            val_correct += preds.eq(labels).sum().item()

    val_acc = 100. * val_correct / val_total
    val_loss /= len(val_loader)
    scheduler.step(val_loss)
    train_loss /= len(train_loader)

    return train_acc, val_acc, train_loss,val_loss

In [ ]:
epochs = 15
best_val_loss = float('inf')
patience = 3
counter = 0
for epoch in range(epochs):
    print(f"Phase 2 Epoch {epoch+1}:")

    train_acc, val_acc, train_loss, val_loss = train_eval_model(
        model, train_loader, val_loader, criterion, optimizer,scheduler
    )
    if val_loss < best_val_loss:
      best_val_loss = val_loss
      counter = 0
      torch.save(model.state_dict(),"best_model.pth")
    else:
      counter += 1
      if counter >= patience:
        print("Early Stopping")
        break
    print(f"Train Acc: {train_acc:.2f} | Val Acc: {val_acc:.2f} | Loss: {train_loss:.2f}")



Phase 2 Epoch 1:
Train Acc: 45.85 | Val Acc: 58.14 | Loss: 2.14
Phase 2 Epoch 2:
Train Acc: 64.03 | Val Acc: 62.79 | Loss: 1.23
Phase 2 Epoch 3:
Train Acc: 68.96 | Val Acc: 65.87 | Loss: 1.03
Phase 2 Epoch 4:
Train Acc: 71.97 | Val Acc: 69.54 | Loss: 0.91
Phase 2 Epoch 5:
Train Acc: 75.99 | Val Acc: 65.57 | Loss: 0.80
Phase 2 Epoch 6:
Train Acc: 78.28 | Val Acc: 67.04 | Loss: 0.69
Phase 2 Epoch 7:
Early Stopping


In [ ]:
optimizer = optim.Adam([{
    'params':model.fc.parameters(),
    'lr':7e-4,
    'weight_decay': 2e-4
},                        {
                            'params':model.layer4.parameters(),
                            'lr':7e-5,
                            'weight_decay':2e-4
                        }
])
model.load_state_dict(torch.load("best_model.pth"))
model.eval()

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

In [ ]:
epochs = 7
best_val_loss = float('inf')
patience = 3
counter = 0
for epoch in range(epochs):
    print(f"Phase 3 Epoch {epoch+1}:")

    train_acc, val_acc, train_loss, val_loss = train_eval_model(
        model, train_loader, val_loader, criterion, optimizer,scheduler
    )
    if val_loss < best_val_loss:
      best_val_loss = val_loss
      counter = 0
      torch.save(model.state_dict(),"best_model.pth")
    else:
      counter += 1
      if counter >= patience:
        print("Early Stopping")
        break
    print(f"Train Acc: {train_acc:.2f} | Val Acc: {val_acc:.2f} | Loss: {train_loss:.2f}")



Phase 3 Epoch 1:
Train Acc: 78.53 | Val Acc: 69.78 | Loss: 0.70
Phase 3 Epoch 2:
Train Acc: 81.29 | Val Acc: 66.60 | Loss: 0.59
Phase 3 Epoch 3:
Train Acc: 83.05 | Val Acc: 68.31 | Loss: 0.54
Phase 3 Epoch 4:


KeyboardInterrupt: 

In [ ]:
optimizer = optim.Adam([{
    'params':model.fc.parameters(),
    'lr':7e-4,
    'weight_decay': 3e-4
},                        {
                            'params':model.layer4.parameters(),
                            'lr':7e-5,
                            'weight_decay':3e-4
                        }
])
model.load_state_dict(torch.load("best_model.pth"))
model.eval()

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

In [ ]:
epochs = 7
best_val_loss = float('inf')
patience = 3
counter = 0
for epoch in range(epochs):
    print(f"Phase 3 Epoch {epoch+1}:")

    train_acc, val_acc, train_loss, val_loss = train_eval_model(
        model, train_loader, val_loader, criterion, optimizer,scheduler
    )
    if val_loss < best_val_loss:
      best_val_loss = val_loss
      counter = 0
      torch.save(model.state_dict(),"best_model.pth")
    else:
      counter += 1
      if counter >= patience:
        print("Early Stopping")
        break
    print(f"Train Acc: {train_acc:.2f} | Val Acc: {val_acc:.2f} | Loss: {train_loss:.2f}")



Phase 3 Epoch 1:
Train Acc: 85.50 | Val Acc: 71.30 | Loss: 0.46
Phase 3 Epoch 2:
Train Acc: 87.44 | Val Acc: 70.76 | Loss: 0.38
Phase 3 Epoch 3:


KeyboardInterrupt: 

In [ ]:
model.load_state_dict(torch.load("best_model.pth"))
model.eval()

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

#Testing

In [ ]:

class TestDogData(Dataset):
    def __init__(self,id_to_idx,cls_to_idx,image_dir,transform=None):
        self.transform = transform

        image_dir = image_dir
        self.paths = []

        for img_name in sorted(os.listdir(image_dir)):


            self.paths.append(os.path.join(image_dir,img_name))

    def __len__(self):
        return len(self.paths)
    def __getitem__(self,idx):

        img_path = self.paths[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)

        return image,img_path

In [ ]:
test_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406],std=[0.229,0.224,0.225])
])
test_image_dir = os.path.join(base_path, "test")
testdataset = TestDogData(id_to_idx,class_to_idx,test_image_dir,test_transform)
test_loader = DataLoader(testdataset,batch_size=32,shuffle=False)

In [ ]:
prediction = []
model.eval()
with torch.no_grad():
        for images, paths in test_loader:
            images = images.to(device)
            outputs = model(images)
            preds = outputs.argmax(dim=1)

            prediction.extend(preds.cpu().numpy())

In [ ]:
idx_to_class = {v:k for k,v in class_to_idx.items()}
pred_labels = [idx_to_class[i] for i in prediction]

In [ ]:
pred_labels[:10]

['japanese_spaniel',
 'samoyed',
 'english_setter',
 'bouvier_des_flandres',
 'tibetan_terrier',
 'tibetan_mastiff',
 'australian_terrier',
 'samoyed',
 'irish_wolfhound',
 'sussex_spaniel']